In [1]:
from src.eval.symantic.generation_utils import get_model 
from src.eval.symantic.embedding_utils import get_documents_embeds

In [2]:
model_id = 'brimmann2/xgemma3-1b-v1'
dataset_id = 'brimmann2/squad_qa1'
split = "train"
XRAG_TOKEN = "<xRAG>"
tokenizer, model = get_model(model_id)
model.set_xrag_token_id(tokenizer.convert_tokens_to_ids(XRAG_TOKEN))
_ = model.eval()
embeddings = get_documents_embeds(dataset_id, split)

`torch_dtype` is deprecated! Use `dtype` instead!


loading generator model...
generator model loaded
loading embedder model and tokenizer...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

embedder model loaded
generating embeddings...
embeddings generated


In [3]:
embeddings[0][0].unsqueeze(0).shape

torch.Size([1, 4096])

In [ ]:
embeddings[0][0].shape

In [5]:
prompt = "<bos><start_of_turn>user\nBackground: <xRAG> Please offer a restatement of the background sentences I've just read.<end_of_turn>\n<start_of_turn>model\n"

In [ ]:
p = "<s>[INST] You're getting across the same point whether you say background: <xRAG> or [/INST]"

In [6]:
input_ids = tokenizer(prompt,return_tensors='pt').input_ids.to("cuda")

In [7]:
input_ids

tensor([[     2,      2,    105,   2364,    107,  11900, 236787, 236743, 262145,
           7323,   2729,    496,   1884,  43861,    529,    506,   1695,  23974,
            564, 236789,    560,   1164,   1676, 236761,    106,    107,    105,
           4368,    107]], device='cuda:0')

In [8]:
generated_output = model.generate(
        input_ids = input_ids,
        do_sample=False,
        max_new_tokens=100,
        pad_token_id=tokenizer.convert_tokens_to_ids("<pad>"),
        retrieval_embeds = embeddings[0][0].unsqueeze(0).to("cuda"),
    )
result = tokenizer.batch_decode(generated_output,skip_special_tokens=True)[0]
print(result)

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The United Nations has been working to promote the adoption of the "Global Health Initiative" (GHI) by 2010. The GHI is a global initiative to promote the health of the world's most vulnerable populations. The initiative is based on the principle that "every child has a right to health." The GHI is supported by the United Nations High Commissioner for Human Rights, the United Nations Development Programme, and the United Nations High Commissioner for Refugees.<end_of_turn>


In [ ]:
result